In [2]:
import mysql.connector
import pandas as pd
from pathlib import Path

conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="qwerty",
    database="cart2insights"
)

cursor = conn.cursor()

print("✅ Connected successfully")

✅ Connected successfully


In [3]:
PROJECT_ROOT = Path.cwd().parent
CLEANED_DATA_PATH = PROJECT_ROOT / "data" / "cleaned"

print("Project root:", PROJECT_ROOT)
print("Cleaned data:", CLEANED_DATA_PATH)

Project root: d:\cart2insights
Cleaned data: d:\cart2insights\data\cleaned


In [4]:
DATASETS = {
    "customers": "olist_customers_dataset.csv",
    "geolocation": "olist_geolocation_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "order_payments": "olist_order_payments_dataset.csv",
    "order_reviews": "olist_order_reviews_dataset.csv",
    "orders": "olist_orders_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "translation": "product_category_name_translation.csv"
}

print("Configured datasets:", len(DATASETS))

Configured datasets: 9


In [5]:
print("=" * 80)
print("CHECKING CLEANED DATASETS")
print("=" * 80)

available_files = {}

for table_name, filename in DATASETS.items():

    file_path = CLEANED_DATA_PATH / filename

    if not file_path.exists():
        print(f"❌ MISSING : {filename}")
        continue

    if file_path.stat().st_size == 0:
        print(f"❌ EMPTY   : {filename}")
        continue

    available_files[table_name] = file_path

    print(
        f"✅ {table_name:15} | "
        f"{file_path.stat().st_size:,} bytes"
    )

print("=" * 80)
print(f"Available: {len(available_files)} / {len(DATASETS)}")
print("=" * 80)

CHECKING CLEANED DATASETS
✅ customers       | 8,661,712 bytes
✅ geolocation     | 44,071,918 bytes
✅ order_items     | 15,024,227 bytes
✅ order_payments  | 5,737,249 bytes
✅ order_reviews   | 14,237,954 bytes
✅ orders          | 16,611,357 bytes
✅ products        | 2,833,152 bytes
✅ sellers         | 166,686 bytes
✅ translation     | 2,612 bytes
Available: 9 / 9


In [6]:
data = {}

for table_name, file_path in available_files.items():

    df = pd.read_csv(
        file_path,
        low_memory=False
    )

    data[table_name] = df

    print(
        f"✅ {table_name:15} | "
        f"Rows: {len(df):,} | "
        f"Columns: {len(df.columns)}"
    )

✅ customers       | Rows: 99,441 | Columns: 5
✅ geolocation     | Rows: 738,332 | Columns: 5
✅ order_items     | Rows: 112,650 | Columns: 7
✅ order_payments  | Rows: 103,886 | Columns: 5
✅ order_reviews   | Rows: 99,224 | Columns: 7
✅ orders          | Rows: 99,441 | Columns: 8
✅ products        | Rows: 32,951 | Columns: 9
✅ sellers         | Rows: 3,095 | Columns: 4
✅ translation     | Rows: 71 | Columns: 2


In [7]:
for table_name, df in data.items():

    print("\n" + "=" * 70)
    print(f"TABLE: {table_name}")
    print("=" * 70)

    print("Rows:", len(df))
    print("Columns:", len(df.columns))

    print("\nColumn names:")

    for column in df.columns:
        print(" -", column)


TABLE: customers
Rows: 99441
Columns: 5

Column names:
 - customer_id
 - customer_unique_id
 - customer_zip_code_prefix
 - customer_city
 - customer_state

TABLE: geolocation
Rows: 738332
Columns: 5

Column names:
 - geolocation_zip_code_prefix
 - geolocation_lat
 - geolocation_lng
 - geolocation_city
 - geolocation_state

TABLE: order_items
Rows: 112650
Columns: 7

Column names:
 - order_id
 - order_item_id
 - product_id
 - seller_id
 - shipping_limit_date
 - price
 - freight_value

TABLE: order_payments
Rows: 103886
Columns: 5

Column names:
 - order_id
 - payment_sequential
 - payment_type
 - payment_installments
 - payment_value

TABLE: order_reviews
Rows: 99224
Columns: 7

Column names:
 - review_id
 - order_id
 - review_score
 - review_comment_title
 - review_comment_message
 - review_creation_date
 - review_answer_timestamp

TABLE: orders
Rows: 99441
Columns: 8

Column names:
 - order_id
 - customer_id
 - order_status
 - order_purchase_timestamp
 - order_approved_at
 - order_de

In [8]:
cursor.execute("SELECT DATABASE()")

current_database = cursor.fetchone()[0]

print("✅ Current MySQL database:", current_database)

✅ Current MySQL database: cart2insights


In [8]:
create_tables_sql = """

CREATE TABLE IF NOT EXISTS geolocation (
    geolocation_zip_code_prefix VARCHAR(10) PRIMARY KEY,
    geolocation_lat FLOAT,
    geolocation_lng FLOAT,
    geolocation_city VARCHAR(100),
    geolocation_state VARCHAR(10)
);

CREATE TABLE IF NOT EXISTS product_category_translation (
    product_category_name VARCHAR(255) PRIMARY KEY,
    product_category_name_english VARCHAR(255)
);

CREATE TABLE IF NOT EXISTS customers (
    customer_id VARCHAR(50) PRIMARY KEY,
    customer_unique_id VARCHAR(50),
    customer_zip_code_prefix VARCHAR(10),
    customer_city VARCHAR(100),
    customer_state VARCHAR(10),

    FOREIGN KEY (customer_zip_code_prefix)
        REFERENCES geolocation(geolocation_zip_code_prefix)
);

CREATE TABLE IF NOT EXISTS sellers (
    seller_id VARCHAR(50) PRIMARY KEY,
    seller_zip_code_prefix VARCHAR(10),
    seller_city VARCHAR(100),
    seller_state VARCHAR(10),

    FOREIGN KEY (seller_zip_code_prefix)
        REFERENCES geolocation(geolocation_zip_code_prefix)
);

CREATE TABLE IF NOT EXISTS products (
    product_id VARCHAR(50) PRIMARY KEY,
    product_category_name VARCHAR(255),
    product_name_length FLOAT,
    product_description_length FLOAT,
    product_photos_qty FLOAT,
    product_weight_g FLOAT,
    product_length_cm FLOAT,
    product_height_cm FLOAT,
    product_width_cm FLOAT,

    FOREIGN KEY (product_category_name)
        REFERENCES product_category_translation(product_category_name)
);

CREATE TABLE IF NOT EXISTS orders (
    order_id VARCHAR(50) PRIMARY KEY,
    customer_id VARCHAR(50),
    order_status VARCHAR(50),
    order_purchase_timestamp DATETIME,
    order_approved_at DATETIME,
    order_delivered_carrier_date DATETIME,
    order_delivered_customer_date DATETIME,
    order_estimated_delivery_date DATETIME,

    FOREIGN KEY (customer_id)
        REFERENCES customers(customer_id)
);

CREATE TABLE IF NOT EXISTS order_items (
    order_id VARCHAR(50),
    order_item_id INT,
    product_id VARCHAR(50),
    seller_id VARCHAR(50),
    shipping_limit_date DATETIME,
    price FLOAT,
    freight_value FLOAT,

    PRIMARY KEY (order_id, order_item_id),

    FOREIGN KEY (order_id)
        REFERENCES orders(order_id),

    FOREIGN KEY (product_id)
        REFERENCES products(product_id),

    FOREIGN KEY (seller_id)
        REFERENCES sellers(seller_id)
);

CREATE TABLE IF NOT EXISTS order_payments (
    order_id VARCHAR(50),
    payment_sequential INT,
    payment_type VARCHAR(50),
    payment_installments INT,
    payment_value FLOAT,

    PRIMARY KEY (
        order_id,
        payment_sequential
    ),

    FOREIGN KEY (order_id)
        REFERENCES orders(order_id)
);

CREATE TABLE IF NOT EXISTS order_reviews (
    review_id VARCHAR(50) PRIMARY KEY,
    order_id VARCHAR(50),
    review_score INT,
    review_comment_title TEXT,
    review_comment_message TEXT,
    review_creation_date DATETIME,
    review_answer_timestamp DATETIME,

    FOREIGN KEY (order_id)
        REFERENCES orders(order_id)
);

"""

for statement in create_tables_sql.split(";"):

    statement = statement.strip()

    if statement:
        cursor.execute(statement)

conn.commit()

print("✅ All MySQL tables created successfully")

✅ All MySQL tables created successfully


In [9]:
cursor.execute("SHOW TABLES")

tables = cursor.fetchall()

print("Tables in cart2insights:\n")

for table in tables:
    print("✅", table[0])

Tables in cart2insights:

✅ customers
✅ geolocation
✅ order_items
✅ order_payments
✅ order_reviews
✅ orders
✅ product_category_translation
✅ products
✅ sellers


In [10]:
df = data["geolocation"].copy()

# Convert ZIP prefix to 5-digit string
df["geolocation_zip_code_prefix"] = (
    pd.to_numeric(
        df["geolocation_zip_code_prefix"],
        errors="coerce"
    )
    .astype("Int64")
    .astype(str)
    .str.zfill(5)
)

# Remove invalid ZIP values
df = df[
    df["geolocation_zip_code_prefix"].notna()
]

# Keep one record per ZIP prefix
df = df.drop_duplicates(
    subset=["geolocation_zip_code_prefix"],
    keep="first"
)

# Replace NaN with None
df = df.astype(object).where(
    pd.notna(df),
    None
)

query = """
INSERT INTO geolocation (
    geolocation_zip_code_prefix,
    geolocation_lat,
    geolocation_lng,
    geolocation_city,
    geolocation_state
)
VALUES (%s, %s, %s, %s, %s)

ON DUPLICATE KEY UPDATE
    geolocation_lat = VALUES(geolocation_lat),
    geolocation_lng = VALUES(geolocation_lng),
    geolocation_city = VALUES(geolocation_city),
    geolocation_state = VALUES(geolocation_state)
"""

rows = list(
    df[
        [
            "geolocation_zip_code_prefix",
            "geolocation_lat",
            "geolocation_lng",
            "geolocation_city",
            "geolocation_state"
        ]
    ].itertuples(
        index=False,
        name=None
    )
)

cursor.executemany(query, rows)
conn.commit()

print(f"✅ Geolocation loaded: {len(rows):,} rows")

✅ Geolocation loaded: 19,015 rows


In [11]:
df = data["translation"].copy()

df = df.drop_duplicates(
    subset=["product_category_name"],
    keep="first"
)

df = df.astype(object).where(
    pd.notna(df),
    None
)

query = """
INSERT INTO product_category_translation (
    product_category_name,
    product_category_name_english
)
VALUES (%s, %s)

ON DUPLICATE KEY UPDATE
    product_category_name_english =
        VALUES(product_category_name_english)
"""

rows = list(
    df[
        [
            "product_category_name",
            "product_category_name_english"
        ]
    ].itertuples(
        index=False,
        name=None
    )
)

cursor.executemany(query, rows)
conn.commit()

print(f"✅ Translation loaded: {len(rows):,} rows")

✅ Translation loaded: 71 rows


In [12]:
df = data["customers"].copy()

df = df.drop_duplicates(
    subset=["customer_id"],
    keep="first"
)

# Normalize ZIP codes
df["customer_zip_code_prefix"] = (
    pd.to_numeric(
        df["customer_zip_code_prefix"],
        errors="coerce"
    )
    .astype("Int64")
    .astype(str)
    .str.zfill(5)
)

# Get valid ZIP codes
cursor.execute("""
    SELECT geolocation_zip_code_prefix
    FROM geolocation
""")

valid_zip_codes = {
    str(row[0]).zfill(5)
    for row in cursor.fetchall()
}

# Keep customers with valid ZIP codes
df = df[
    df["customer_zip_code_prefix"].isin(
        valid_zip_codes
    )
]

df = df.astype(object).where(
    pd.notna(df),
    None
)

query = """
INSERT INTO customers (
    customer_id,
    customer_unique_id,
    customer_zip_code_prefix,
    customer_city,
    customer_state
)
VALUES (%s, %s, %s, %s, %s)

ON DUPLICATE KEY UPDATE
    customer_unique_id = VALUES(customer_unique_id),
    customer_zip_code_prefix = VALUES(customer_zip_code_prefix),
    customer_city = VALUES(customer_city),
    customer_state = VALUES(customer_state)
"""

rows = list(
    df[
        [
            "customer_id",
            "customer_unique_id",
            "customer_zip_code_prefix",
            "customer_city",
            "customer_state"
        ]
    ].itertuples(
        index=False,
        name=None
    )
)

cursor.executemany(query, rows)
conn.commit()

print(f"✅ Customers loaded: {len(rows):,} rows")

✅ Customers loaded: 99,163 rows


In [13]:
df = data["sellers"].copy()

df = df.drop_duplicates(
    subset=["seller_id"],
    keep="first"
)

# Normalize ZIP codes
df["seller_zip_code_prefix"] = (
    pd.to_numeric(
        df["seller_zip_code_prefix"],
        errors="coerce"
    )
    .astype("Int64")
    .astype(str)
    .str.zfill(5)
)

cursor.execute("""
    SELECT geolocation_zip_code_prefix
    FROM geolocation
""")

valid_zip_codes = {
    str(row[0]).zfill(5)
    for row in cursor.fetchall()
}

df = df[
    df["seller_zip_code_prefix"].isin(
        valid_zip_codes
    )
]

df = df.astype(object).where(
    pd.notna(df),
    None
)

query = """
INSERT INTO sellers (
    seller_id,
    seller_zip_code_prefix,
    seller_city,
    seller_state
)
VALUES (%s, %s, %s, %s)

ON DUPLICATE KEY UPDATE
    seller_zip_code_prefix =
        VALUES(seller_zip_code_prefix),
    seller_city =
        VALUES(seller_city),
    seller_state =
        VALUES(seller_state)
"""

rows = list(
    df[
        [
            "seller_id",
            "seller_zip_code_prefix",
            "seller_city",
            "seller_state"
        ]
    ].itertuples(
        index=False,
        name=None
    )
)

cursor.executemany(query, rows)
conn.commit()

print(f"✅ Sellers loaded: {len(rows):,} rows")

✅ Sellers loaded: 3,088 rows


In [14]:
df = data["products"].copy()

df = df.drop_duplicates(
    subset=["product_id"],
    keep="first"
)

cursor.execute("""
    SELECT product_category_name
    FROM product_category_translation
""")

valid_categories = {
    row[0]
    for row in cursor.fetchall()
}

df = df[
    df["product_category_name"].isna()
    |
    df["product_category_name"].isin(
        valid_categories
    )
]

df = df.astype(object).where(
    pd.notna(df),
    None
)

query = """
INSERT INTO products (
    product_id,
    product_category_name,
    product_name_length,
    product_description_length,
    product_photos_qty,
    product_weight_g,
    product_length_cm,
    product_height_cm,
    product_width_cm
)
VALUES (
    %s, %s, %s, %s, %s,
    %s, %s, %s, %s
)

ON DUPLICATE KEY UPDATE
    product_category_name =
        VALUES(product_category_name),
    product_name_length =
        VALUES(product_name_length),
    product_description_length =
        VALUES(product_description_length),
    product_photos_qty =
        VALUES(product_photos_qty),
    product_weight_g =
        VALUES(product_weight_g),
    product_length_cm =
        VALUES(product_length_cm),
    product_height_cm =
        VALUES(product_height_cm),
    product_width_cm =
        VALUES(product_width_cm)
"""

rows = list(
    df[
        [
            "product_id",
            "product_category_name",
            "product_name_length",
            "product_description_length",
            "product_photos_qty",
            "product_weight_g",
            "product_length_cm",
            "product_height_cm",
            "product_width_cm"
        ]
    ].itertuples(
        index=False,
        name=None
    )
)

cursor.executemany(query, rows)
conn.commit()

print(f"✅ Products loaded: {len(rows):,} rows")

✅ Products loaded: 32,328 rows


In [15]:
df = data["orders"].copy()

df = df.drop_duplicates(
    subset=["order_id"],
    keep="first"
)

# Get valid customer IDs
cursor.execute("""
    SELECT customer_id
    FROM customers
""")

valid_customer_ids = {
    str(row[0])
    for row in cursor.fetchall()
}

df["customer_id"] = df["customer_id"].astype(str)

df = df[
    df["customer_id"].isin(
        valid_customer_ids
    )
]

# Convert dates
date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for column in date_columns:
    df[column] = pd.to_datetime(
        df[column],
        errors="coerce"
    )

df = df.astype(object).where(
    pd.notna(df),
    None
)

query = """
INSERT INTO orders (
    order_id,
    customer_id,
    order_status,
    order_purchase_timestamp,
    order_approved_at,
    order_delivered_carrier_date,
    order_delivered_customer_date,
    order_estimated_delivery_date
)
VALUES (
    %s, %s, %s, %s,
    %s, %s, %s, %s
)

ON DUPLICATE KEY UPDATE
    customer_id =
        VALUES(customer_id),
    order_status =
        VALUES(order_status),
    order_purchase_timestamp =
        VALUES(order_purchase_timestamp),
    order_approved_at =
        VALUES(order_approved_at),
    order_delivered_carrier_date =
        VALUES(order_delivered_carrier_date),
    order_delivered_customer_date =
        VALUES(order_delivered_customer_date),
    order_estimated_delivery_date =
        VALUES(order_estimated_delivery_date)
"""

rows = list(
    df[
        [
            "order_id",
            "customer_id",
            "order_status",
            "order_purchase_timestamp",
            "order_approved_at",
            "order_delivered_carrier_date",
            "order_delivered_customer_date",
            "order_estimated_delivery_date"
        ]
    ].itertuples(
        index=False,
        name=None
    )
)

cursor.executemany(query, rows)
conn.commit()

print(f"✅ Orders loaded: {len(rows):,} rows")

✅ Orders loaded: 99,163 rows


In [16]:
df = data["order_items"].copy()

# Remove duplicate composite keys
df = df.drop_duplicates(
    subset=["order_id", "order_item_id"],
    keep="first"
)

# Get valid IDs
cursor.execute("SELECT order_id FROM orders")
valid_orders = {
    str(row[0])
    for row in cursor.fetchall()
}

cursor.execute("SELECT product_id FROM products")
valid_products = {
    str(row[0])
    for row in cursor.fetchall()
}

cursor.execute("SELECT seller_id FROM sellers")
valid_sellers = {
    str(row[0])
    for row in cursor.fetchall()
}

df["order_id"] = df["order_id"].astype(str)
df["product_id"] = df["product_id"].astype(str)
df["seller_id"] = df["seller_id"].astype(str)

df = df[
    df["order_id"].isin(valid_orders)
    &
    df["product_id"].isin(valid_products)
    &
    df["seller_id"].isin(valid_sellers)
]

df["shipping_limit_date"] = pd.to_datetime(
    df["shipping_limit_date"],
    errors="coerce"
)

df = df.astype(object).where(
    pd.notna(df),
    None
)

query = """
INSERT INTO order_items (
    order_id,
    order_item_id,
    product_id,
    seller_id,
    shipping_limit_date,
    price,
    freight_value
)
VALUES (
    %s, %s, %s, %s,
    %s, %s, %s
)

ON DUPLICATE KEY UPDATE
    product_id =
        VALUES(product_id),
    seller_id =
        VALUES(seller_id),
    shipping_limit_date =
        VALUES(shipping_limit_date),
    price =
        VALUES(price),
    freight_value =
        VALUES(freight_value)
"""

rows = list(
    df[
        [
            "order_id",
            "order_item_id",
            "product_id",
            "seller_id",
            "shipping_limit_date",
            "price",
            "freight_value"
        ]
    ].itertuples(
        index=False,
        name=None
    )
)

cursor.executemany(query, rows)
conn.commit()

print(f"✅ Order items loaded: {len(rows):,} rows")

✅ Order items loaded: 110,476 rows


In [17]:
df = data["order_payments"].copy()

df = df.drop_duplicates(
    subset=[
        "order_id",
        "payment_sequential"
    ],
    keep="first"
)

cursor.execute(
    "SELECT order_id FROM orders"
)

valid_orders = {
    str(row[0])
    for row in cursor.fetchall()
}

df["order_id"] = df["order_id"].astype(str)

df = df[
    df["order_id"].isin(valid_orders)
]

df = df.astype(object).where(
    pd.notna(df),
    None
)

query = """
INSERT INTO order_payments (
    order_id,
    payment_sequential,
    payment_type,
    payment_installments,
    payment_value
)
VALUES (%s, %s, %s, %s, %s)

ON DUPLICATE KEY UPDATE
    payment_type =
        VALUES(payment_type),
    payment_installments =
        VALUES(payment_installments),
    payment_value =
        VALUES(payment_value)
"""

rows = list(
    df[
        [
            "order_id",
            "payment_sequential",
            "payment_type",
            "payment_installments",
            "payment_value"
        ]
    ].itertuples(
        index=False,
        name=None
    )
)

cursor.executemany(query, rows)
conn.commit()

print(f"✅ Payments loaded: {len(rows):,} rows")

✅ Payments loaded: 103,599 rows


In [18]:
df = data["order_reviews"].copy()

df = df.drop_duplicates(
    subset=["review_id"],
    keep="first"
)

cursor.execute(
    "SELECT order_id FROM orders"
)

valid_orders = {
    str(row[0])
    for row in cursor.fetchall()
}

df["order_id"] = df["order_id"].astype(str)

df = df[
    df["order_id"].isin(valid_orders)
]

date_columns = [
    "review_creation_date",
    "review_answer_timestamp"
]

for column in date_columns:
    df[column] = pd.to_datetime(
        df[column],
        errors="coerce"
    )

df = df.astype(object).where(
    pd.notna(df),
    None
)

query = """
INSERT INTO order_reviews (
    review_id,
    order_id,
    review_score,
    review_comment_title,
    review_comment_message,
    review_creation_date,
    review_answer_timestamp
)
VALUES (
    %s, %s, %s, %s,
    %s, %s, %s
)

ON DUPLICATE KEY UPDATE
    order_id =
        VALUES(order_id),
    review_score =
        VALUES(review_score),
    review_comment_title =
        VALUES(review_comment_title),
    review_comment_message =
        VALUES(review_comment_message),
    review_creation_date =
        VALUES(review_creation_date),
    review_answer_timestamp =
        VALUES(review_answer_timestamp)
"""

rows = list(
    df[
        [
            "review_id",
            "order_id",
            "review_score",
            "review_comment_title",
            "review_comment_message",
            "review_creation_date",
            "review_answer_timestamp"
        ]
    ].itertuples(
        index=False,
        name=None
    )
)

cursor.executemany(query, rows)
conn.commit()

print(f"✅ Reviews loaded: {len(rows):,} rows")

✅ Reviews loaded: 98,134 rows


In [19]:
checks = {
    "Customers → Geolocation": """
        SELECT COUNT(*)
        FROM customers c
        LEFT JOIN geolocation g
            ON c.customer_zip_code_prefix =
               g.geolocation_zip_code_prefix
        WHERE g.geolocation_zip_code_prefix IS NULL
    """,

    "Sellers → Geolocation": """
        SELECT COUNT(*)
        FROM sellers s
        LEFT JOIN geolocation g
            ON s.seller_zip_code_prefix =
               g.geolocation_zip_code_prefix
        WHERE g.geolocation_zip_code_prefix IS NULL
    """,

    "Orders → Customers": """
        SELECT COUNT(*)
        FROM orders o
        LEFT JOIN customers c
            ON o.customer_id = c.customer_id
        WHERE c.customer_id IS NULL
    """,

    "Items → Orders": """
        SELECT COUNT(*)
        FROM order_items oi
        LEFT JOIN orders o
            ON oi.order_id = o.order_id
        WHERE o.order_id IS NULL
    """,

    "Items → Products": """
        SELECT COUNT(*)
        FROM order_items oi
        LEFT JOIN products p
            ON oi.product_id = p.product_id
        WHERE p.product_id IS NULL
    """,

    "Items → Sellers": """
        SELECT COUNT(*)
        FROM order_items oi
        LEFT JOIN sellers s
            ON oi.seller_id = s.seller_id
        WHERE s.seller_id IS NULL
    """
}

print("=" * 70)
print("FOREIGN KEY VALIDATION")
print("=" * 70)

for name, query in checks.items():

    cursor.execute(query)

    invalid = cursor.fetchone()[0]

    if invalid == 0:
        print(f"✅ {name}: 0 invalid")
    else:
        print(f"❌ {name}: {invalid:,} invalid")

FOREIGN KEY VALIDATION
✅ Customers → Geolocation: 0 invalid
✅ Sellers → Geolocation: 0 invalid
✅ Orders → Customers: 0 invalid
✅ Items → Orders: 0 invalid
✅ Items → Products: 0 invalid
✅ Items → Sellers: 0 invalid


In [20]:
orders = pd.read_sql(
    "SELECT * FROM orders",
    conn
)

order_items = pd.read_sql(
    "SELECT * FROM order_items",
    conn
)

customers = pd.read_sql(
    "SELECT * FROM customers",
    conn
)

sellers = pd.read_sql(
    "SELECT * FROM sellers",
    conn
)

products = pd.read_sql(
    "SELECT * FROM products",
    conn
)

payments = pd.read_sql(
    "SELECT * FROM order_payments",
    conn
)

reviews = pd.read_sql(
    "SELECT * FROM order_reviews",
    conn
)

print("Orders:", orders.shape)
print("Order Items:", order_items.shape)
print("Customers:", customers.shape)
print("Sellers:", sellers.shape)
print("Products:", products.shape)
print("Payments:", payments.shape)
print("Reviews:", reviews.shape)

C:\Users\Asus-2025\AppData\Local\Temp\ipykernel_25632\873284673.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  orders = pd.read_sql(
C:\Users\Asus-2025\AppData\Local\Temp\ipykernel_25632\873284673.py:6: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  order_items = pd.read_sql(
C:\Users\Asus-2025\AppData\Local\Temp\ipykernel_25632\873284673.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  customers = pd.read_sql(
C:\Users\Asus-2025\AppData\Local\Temp\ipykernel_25632\873284673.py:16: UserWarning: pandas only supports S

Orders: (99163, 8)
Order Items: (110476, 7)
Customers: (99163, 5)
Sellers: (3088, 4)
Products: (32328, 9)
Payments: (103599, 5)
Reviews: (98134, 7)


In [21]:
cursor.execute("""
    SELECT
        order_status,
        COUNT(*) AS order_count
    FROM orders
    GROUP BY order_status
    ORDER BY order_count DESC
""")

results = cursor.fetchall()

print("Orders by status:\n")

for row in results:
    print(row)

Orders by status:

('delivered', 96214)
('shipped', 1100)
('canceled', 623)
('unavailable', 607)
('invoiced', 313)
('processing', 300)
('created', 4)
('approved', 2)


In [22]:
cursor.execute("""
    SELECT
        ROUND(SUM(payment_value), 2) AS total_revenue
    FROM order_payments
""")

result = cursor.fetchone()

print(
    "Total Revenue:",
    result[0]
)

Total Revenue: 15964050.25


In [23]:
cursor.execute("""
    SELECT
        ROUND(
            SUM(payment_value)
            / COUNT(DISTINCT order_id),
            2
        ) AS average_order_value
    FROM order_payments
""")

result = cursor.fetchone()

print(
    "Average Order Value:",
    result[0]
)

Average Order Value: 160.99


In [24]:
cursor.execute("""
    SELECT
        COALESCE(
            p.product_category_name,
            'Unknown'
        ) AS category,
        COUNT(*) AS items_sold
    FROM order_items oi
    LEFT JOIN products p
        ON oi.product_id = p.product_id
    GROUP BY category
    ORDER BY items_sold DESC
    LIMIT 10
""")

results = cursor.fetchall()

print("Top 10 product categories:\n")

for row in results:
    print(row)

Top 10 product categories:

('cama_mesa_banho', 11087)
('beleza_saude', 9641)
('esporte_lazer', 8624)
('moveis_decoracao', 8315)
('informatica_acessorios', 7801)
('utilidades_domesticas', 6945)
('relogios_presentes', 5968)
('telefonia', 4517)
('ferramentas_jardim', 4336)
('automotivo', 4226)


In [25]:
cursor.execute("""
    SELECT
        oi.seller_id,
        ROUND(
            SUM(oi.price + oi.freight_value),
            2
        ) AS seller_revenue
    FROM order_items oi
    GROUP BY oi.seller_id
    ORDER BY seller_revenue DESC
    LIMIT 10
""")

results = cursor.fetchall()

print("Top 10 sellers by revenue:\n")

for row in results:
    print(row)

Top 10 sellers by revenue:

('4869f7a5dfa277a7dca6462dcf3b52b2', 249449.02)
('7c67e1448b00f6e969d365cea6b010ab', 239038.74)
('4a3ca9315b744ce9f8e9374361493884', 235303.85)
('53243585a1d6dc2643021fd1853d8905', 233948.51)
('fa1c13f2614d7b5c4749cbc52fecda94', 202339.02)
('da8622b14eb17ae2831f4ac5b9dab84a', 185004.46)
('7e93a43ef30c4f03f38b393420bc753a', 181282.04)
('1025f0e2d44d7041d6cf58b6550e0bfa', 172089.59)
('7a67c85e85bb2ce8582c35f2203ad736', 162221.66)
('955fee9216a65b617aa5c0531780ce60', 160119.49)


In [26]:
cursor.execute("""
    SELECT
        c.customer_unique_id,
        ROUND(
            SUM(op.payment_value),
            2
        ) AS total_spending
    FROM customers c
    JOIN orders o
        ON c.customer_id = o.customer_id
    JOIN order_payments op
        ON o.order_id = op.order_id
    GROUP BY c.customer_unique_id
    ORDER BY total_spending DESC
    LIMIT 10
""")

results = cursor.fetchall()

print("Top 10 customers by spending:\n")

for row in results:
    print(row)

Top 10 customers by spending:

('0a0a92112bd4c708ca5fde585afaa872', 13664.08)
('46450c74a0d8c5ca9395da1daac6c120', 9553.02)
('da122df9eeddfedc1dc1f5349a1a690c', 7571.63)
('763c8b1c9c68a0229c42c9fc6f662b93', 7274.88)
('dc4802a71eae9be1dd28f5d788ceb526', 6929.31)
('459bef486812aa25204be022145caa62', 6922.21)
('ff4159b92c40ebe40454e3e6a7c35ed6', 6726.66)
('4007669dec559734d6f53e029e360987', 6081.54)
('5d0a2980b292d049061542014e8960bf', 4809.44)
('eebb5dda148d3893cdaf5b5ca3040ccb', 4764.34)


In [27]:
cursor.close()
conn.close()

print("✅ MySQL connection closed")

✅ MySQL connection closed
